# 💒 Multi-Agent Coordinator: Event Planning with Specialized Subagents

### Overview & Architecture
Complex real-world tasks often overwhelm a single prompt or monolithic agent. The **Hierarchical Multi-Agent Coordinator** pattern solves this by dividing responsibilities:
1. **Supervisor / Coordinator Agent**: Understands high-level user intent, extracts parameters into a shared state, and delegates tasks to domain-specialist agents.
2. **Specialized Subagents**: Each possesses its own dedicated prompt, constraints, and tool ecosystem:
   - **Travel Agent**: Interfaces with travel servers via MCP.
   - **Venue Agent**: Executes iterative search queries using Tavily.
   - **Playlist Agent**: Queries a relational SQLite music database (Chinook).
3. **Groq Acceleration**: All agents are powered by Groq's high-speed **`llama-3.3-70b-versatile`** model, enabling rapid parallel and sequential agent reasoning.

## 📐 System Architecture: Coordinator & Subagent Hierarchy

The diagram below illustrates the hierarchical coordinator pattern:

<div align="center">
  <img src="images/03_multi_agent_coordinator.png" alt="Hierarchical Multi-Agent Coordinator Architecture" width="100%" />
</div>

<br/>

<details>
<summary><b>🔍 View Raw Mermaid Diagram Syntax</b></summary>

```mermaid
flowchart TD
    User([👤 User Request]) --> Coord[👑 Wedding Coordinator Agent<br/>Groq llama-3.3-70b-versatile]
    Coord <--> State[(📋 Shared WeddingState<br/>origin, destination, guest_count, genre)]

    Coord -->|Tool Call: search_flights| Travel[✈️ Travel Subagent<br/>Groq llama-3.3-70b-versatile]
    Coord -->|Tool Call: search_venues| Venue[🏰 Venue Subagent<br/>Groq llama-3.3-70b-versatile]
    Coord -->|Tool Call: suggest_playlist| Music[🎷 Playlist Subagent<br/>Groq llama-3.3-70b-versatile]

    Travel <-->|HTTP JSON-RPC + Retry Interceptor| KiwiMCP[🌐 Kiwi Travel MCP Server]
    Venue <-->|Web Search API| Tavily[🔍 Tavily Web Search]
    Music <-->|SQL Queries| DB[(🗄️ Chinook SQLite Database)]

    Travel -->|Flight Shortlist| Coord
    Venue -->|Venue Proposals| Coord
    Music -->|Curated Tracklist| Coord

    Coord -->|Synthesized Master Plan| Output([💍 Comprehensive Wedding Plan])
```

</details>


## 1. Environment & Key Verification

Load environment variables and verify credentials for Groq and Tavily.

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

print("Checking environment keys:")
print(f"- GROQ_API_KEY present: {bool(os.getenv('GROQ_API_KEY'))}")
print(f"- TAVILY_API_KEY present: {bool(os.getenv('TAVILY_API_KEY'))}")

## 2. Tool Setup: MCP Travel Client with Resilient Retry Interceptor

Networked MCP servers can experience transient timeouts or rate limits. We implement a `RetryMCPInterceptor` that:
- Catches retryable MCP error codes (e.g. `-32603`).
- Applies exponential backoff (`2 ** attempt`).
- Intercepts non-retryable errors and surfaces friendly error text to the agent rather than crashing the execution loop.

In [ ]:
import asyncio
from langchain_mcp_adapters.client import MultiServerMCPClient
from mcp.shared.exceptions import McpError
from mcp.types import CallToolResult, TextContent

RETRYABLE_MCP_CODES = {-32603}

class RetryMCPInterceptor:
    """Intercept MCP tool calls: retry transient failures, surface all errors gracefully.

    - Retryable McpError codes (e.g. -32603): retry with exponential backoff.
    - Non-retryable McpError codes (e.g. -32602): return error message immediately.
    - Any other exception (fetch failed, network errors, etc.): retry then return error message.
    """

    def __init__(self, max_retries: int = 3):
        self.max_retries = max_retries

    async def __call__(self, request, handler):
        last_error = None
        for attempt in range(self.max_retries):
            try:
                return await handler(request)
            except McpError as exc:
                last_error = exc
                print(f"[MCP interceptor] {type(exc).__name__} on {request.name} "
                      f"(code {exc.error.code}, attempt {attempt+1}/{self.max_retries}): {exc}")
                if exc.error.code not in RETRYABLE_MCP_CODES:
                    return CallToolResult(
                        content=[TextContent(type="text", text=f"Tool call failed (non-retryable): {exc}")],
                        isError=False,
                    )
            except Exception as exc:
                last_error = exc
                print(f"[MCP interceptor] {type(exc).__name__} on {request.name} "
                      f"(attempt {attempt+1}/{self.max_retries}): {exc}")

            if attempt < self.max_retries - 1:
                await asyncio.sleep(2 ** attempt)

        print(f"[MCP interceptor] all {self.max_retries} retries exhausted for {request.name}")
        return CallToolResult(
            content=[TextContent(type="text", text=f"Tool call failed after {self.max_retries} attempts: {last_error}")],
            isError=False,
        )

client = MultiServerMCPClient(
    {
        "travel_server": {
            "transport": "streamable_http",
            "url": "https://mcp.kiwi.com"
        }
    },
    tool_interceptors=[RetryMCPInterceptor()],
)

tools = await client.get_tools()
print(f"Discovered {len(tools)} travel tools from Kiwi MCP.")

## 3. Tool Setup: Venue Search with Search Budgeting

Agents in loops can get trapped in repetitive searches. Here we enforce a search budget: the agent must pass `search_number` and `max_search_number` to ensure termination.

In [ ]:
from typing import Dict, Any
from tavily import TavilyClient
from langchain.tools import tool

tavily_client = TavilyClient()

@tool
def web_search(query: str, search_number: int, max_search_number: int) -> Dict[str, Any]:
    """Search the web for information. You must track your search count by providing
    search_number (starting at 1) and max_search_number on every call.
    Queries must use only plain text characters. Do not use accented or special characters     
      (e.g., use 'capacite' instead of 'capacité').
    """
    if search_number > max_search_number:
        return {"message": "Search limit reached. Please summarize your findings and provide your final answer."}
    try:
        return tavily_client.search(query)
    except Exception as e:
        return {"error": str(e)}

## 4. Tool Setup: Relational Database Querying

We connect to a local SQLite database (`Chinook.db`) using LangChain's `SQLDatabase` utility to let our playlist agent explore musical tracks, genres, and artists.

In [ ]:
from langchain_community.utilities import SQLDatabase

db = SQLDatabase.from_uri("sqlite:///resources/Chinook.db")

@tool
def query_playlist_db(query: str) -> str:
    """Query the database for playlist information, genres, artists, and tracks."""
    try:
        return db.run(query)
    except Exception as e:
        return f"Error querying database: {e}"

## 5. Defining the Shared Coordinator State

We define `WeddingState` extending LangChain's base `AgentState`. This shared schema holds event parameters extracted by the coordinator from natural language.

In [ ]:
from langchain.agents import AgentState

class WeddingState(AgentState):
    origin: str
    destination: str
    guest_count: str
    genre: str

## 6. Building the 3 Specialized Subagents with Groq

Each subagent is initialized with `ChatGroq(model="llama-3.3-70b-versatile")`, customized with strict instructions and bounded tools:
1. **Travel Agent**: Uses Kiwi MCP flight search tools.
2. **Venue Agent**: Uses budgeted web search for wedding venues matching capacity.
3. **Playlist Agent**: Uses SQLite queries to curate a playlist matching the chosen genre.

In [ ]:
from langchain_groq import ChatGroq
from langchain.agents import create_agent

# Shared Groq chat model for all agents
groq_llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0.1
)

# 1. Travel Agent
travel_agent = create_agent(
    model=groq_llm,
    tools=tools,
    system_prompt="""
    You are a travel agent. Search for flights to the desired destination wedding location.
    You are not allowed to ask any more follow up questions, you must find the best flight options based on the following criteria:
    - Price (lowest, economy class)
    - Duration (shortest)
    - Date (time of year which you believe is best for a wedding at this location)
    To make things easy, only look for one ticket, one way.
    You may need to make multiple searches to iteratively find the best options.
    You will be given no extra information, only the origin and destination. It is your job to think critically about the best options.
    If the MCP tool fails, returns malformed output, or does not give you usable flight results, try the tool again.
    Once you have found the best options, let the user know your shortlist of options.
    """
)

In [ ]:
# 2. Venue Agent
venue_agent = create_agent(
    model=groq_llm,
    tools=[web_search],
    system_prompt="""
    You are a venue specialist. Search for venues in the desired location, and with the desired capacity.
    You are not allowed to ask any more follow up questions, you must find the best venue options based on the following criteria:
    - Price (lowest)
    - Capacity (exact match)
    - Reviews (highest)
    You may need to make multiple searches to iteratively find the best options. 
    You have a suggested limit of 12 web searches. Count every web_search call you make.
    After 12 searches, you should stop searching and summarize the best options you have
    found so far.
    """
)

In [ ]:
# 3. Playlist Agent
playlist_agent = create_agent(
    model=groq_llm,
    tools=[query_playlist_db],
    system_prompt="""
    You are a music director. Query the Chinook SQLite database to curate a wedding reception playlist.
    Find tracks matching the requested genre.
    Once you have found a shortlist of tracks, present a playlist of ~20 tracks tailored to the wedding vibe.
    """
)

## 7. Converting Subagents into Coordinator Tools

In LangChain, a subagent can be exposed as a `@tool` for the master coordinator. The coordinator simply invokes `await subagent.ainvoke(...)` as if it were calling an API function!

In [ ]:
from langchain.tools import ToolRuntime
from langchain.messages import HumanMessage
from langgraph.types import Command

@tool
async def search_flights(runtime: ToolRuntime) -> str:
    """Call the travel agent to search for flights based on state."""
    prompt = f"Find flights from {runtime.state.get('origin')} to {runtime.state.get('destination')}"
    res = await travel_agent.ainvoke({"messages": [HumanMessage(content=prompt)]})
    return res["messages"][-1].content

@tool
async def search_venues(runtime: ToolRuntime) -> str:
    """Call the venue specialist to search for venues based on state."""
    prompt = f"Find venues in {runtime.state.get('destination')} for {runtime.state.get('guest_count')} guests"
    res = await venue_agent.ainvoke({"messages": [HumanMessage(content=prompt)]})
    return res["messages"][-1].content

@tool
async def suggest_playlist(runtime: ToolRuntime) -> str:
    """Call the music director to create a playlist based on state."""
    prompt = f"Suggest a {runtime.state.get('genre')} wedding playlist"
    res = await playlist_agent.ainvoke({"messages": [HumanMessage(content=prompt)]})
    return res["messages"][-1].content

@tool
def update_state(origin: str, destination: str, guest_count: str, genre: str) -> Command:
    """Update the wedding state with user preferences."""
    return Command(update={
        "origin": origin,
        "destination": destination,
        "guest_count": guest_count,
        "genre": genre
    })

## 8. Compiling the Coordinator Agent

The coordinator is initialized with Groq, given access to the 4 tools (`update_state`, `search_flights`, `search_venues`, `suggest_playlist`), and linked to the `WeddingState` schema.

In [ ]:
coordinator_prompt = """
You are a master wedding coordinator. Your role is to help clients plan their destination wedding.

1. First, extract the origin, destination, guest count, and music genre from the user's request and update the state.
2. Next, coordinate with your specialist subagents:
   - Call `search_flights` to find travel options.
   - Call `search_venues` to locate suitable venues.
   - Call `suggest_playlist` to build the reception playlist.
3. Finally, compile and synthesize all specialist findings into a beautiful, cohesive master wedding itinerary.
"""

coordinator_agent = create_agent(
    model=groq_llm,
    tools=[update_state, search_flights, search_venues, suggest_playlist],
    system_prompt=coordinator_prompt,
    state_schema=WeddingState
)
print("✅ Master Coordinator Agent compiled successfully.")

## 9. Executing the Multi-Agent Workflow

We invoke the coordinator with a multi-faceted user request: planning a 100-guest wedding in Paris with jazz music originating from London.

In [ ]:
user_prompt = "I want to plan a wedding in Paris for 100 guests with a jazz vibe. We will be traveling from London."

response = await coordinator_agent.ainvoke(
    {"messages": [HumanMessage(content=user_prompt)]}
)

print("\n=================== MASTER WEDDING PLAN ===================\n")
print(response["messages"][-1].content)